# AUC calculation per fold and repeat

In [1]:
# Load the pred_age and true_age from the CSV file and calculate AUC
import pandas as pd
import os
import numpy as np

FOLD = 7
REPEAT = 10

# Load metadata
METADATA_PATH = "../../../AD_DECODE_data3.xlsx"
metadata_df = pd.read_excel(METADATA_PATH, sheet_name='AD_DECODE_data2')

# Convert 'MRI_Exam' to match 'subject_id' format
metadata_df['subject_id'] = metadata_df['MRI_Exam'].apply(lambda x: f"{int(x):05d}" if pd.notnull(x) else np.nan)


In [9]:
# Calculate AUC for each fold and repeat
def calculate_auc_per_fold_repeat(fold, repeat):
    # Load the results for the specific fold and repeat
    test_results_path = f"column_md_results/fold_{fold}_rep_{repeat}_test_results.csv"
    train_results_path = f"column_md_results/fold_{fold}_rep_{repeat}_train_results.csv"
    if not os.path.exists(test_results_path) or not os.path.exists(train_results_path):
        print(f"Results file for fold {fold}, repeat {repeat} does not exist.")
        return None
    
    df_test = pd.read_csv(test_results_path)
    df_train = pd.read_csv(train_results_path)
    
    # Merge with metadata to get true y_trues
    # Ensure 'subject_id' is in the correct format
    df_test['subject_id'] = df_test['subject_id'].apply(lambda x: f"{int(x):05d}" if pd.notnull(x) else np.nan)
    df_train['subject_id'] = df_train['subject_id'].apply(lambda x: f"{int(x):05d}" if pd.notnull(x) else np.nan)
    df_test = df_test.merge(metadata_df[['subject_id', 'risk_for_ad']], on='subject_id', how='left')
    df_train = df_train.merge(metadata_df[['subject_id', 'risk_for_ad']], on='subject_id', how='left')

    # Drop rows with NaN in 'y_pred' or 'y_true'
    df_test = df_test.dropna(subset=['y_pred', 'y_true'])
    df_train = df_train.dropna(subset=['y_pred', 'y_true'])

    # Fit linear regression in training data: BAG ~ y_true
    from sklearn.linear_model import LinearRegression
    model = LinearRegression()
    X_train = df_train[['y_pred']]
    y_train = df_train['y_true']
    reg = model.fit(X_train, y_train)
    # Predict y_true effect in test data
    y_true_effect = reg.predict(df_test[['y_pred']])
    df_test['cBAG'] = df_test['y_pred'] - y_true_effect

    # Group the test subjects into 2 groups by 'risk_for_ad'= 0 and 1 or 'risk_for_ad'= 2 and 3
    risk_group_01 = df_test[df_test['risk_for_ad'].isin([0, 1])]
    risk_group_23 = df_test[df_test['risk_for_ad'].isin([2, 3])]

    # Binarize the 'risk_for_ad' column
    y = np.where(risk_group_01['risk_for_ad'] == 0, 0, 1)
    
    from sklearn.metrics import roc_auc_score
    auc = roc_auc_score(y, df_test['cBAG'])
    # Compute the confidence interval for AUC
    from sklearn.utils import resample
    n_iterations = 1000
    auc_scores = []
    for _ in range(n_iterations):
        # Bootstrap resample
        sample = resample(df_test, replace=True)
        y_sample = np.where(sample['risk_for_ad'].isin([0, 1]), 0, 1)
        # Skip if both classes are not present
        if len(np.unique(y_sample)) < 2:
            continue
        auc_sample = roc_auc_score(y_sample, sample['cBAG'])
        auc_scores.append(auc_sample)
    if len(auc_scores) > 0:
        lower_bound = np.percentile(auc_scores, 2.5)
        upper_bound = np.percentile(auc_scores, 97.5)
    else:
        lower_bound = np.nan
        upper_bound = np.nan
    print(f"Fold {fold}, Repeat {repeat}: AUC = {auc:.4f}, 95% CI = [{lower_bound:.4f}, {upper_bound:.4f}]")

    # Save the results to a CSV file
    results_df = pd.DataFrame({
        'fold': [fold],
        'repeat': [repeat],
        'auc': [auc],
        'lower_bound': [lower_bound],
        'upper_bound': [upper_bound]
    })
    results_df.to_csv(f"column_md_results/fold_{fold}_rep_{repeat}_auc_results.csv", index=False)
    
    return auc

# Calculate AUC for all folds and repeats

for fold in range(1, FOLD + 1):
    for repeat in range(1, REPEAT + 1):
        calculate_auc_per_fold_repeat(fold, repeat)

Fold 1, Repeat 1: AUC = 0.4762, 95% CI = [nan, nan]
Fold 1, Repeat 2: AUC = 0.4286, 95% CI = [nan, nan]
Fold 1, Repeat 3: AUC = 0.5238, 95% CI = [nan, nan]
Fold 1, Repeat 4: AUC = 0.4286, 95% CI = [nan, nan]
Fold 1, Repeat 5: AUC = 0.4762, 95% CI = [nan, nan]
Fold 1, Repeat 6: AUC = 0.4762, 95% CI = [nan, nan]
Fold 1, Repeat 7: AUC = 0.4286, 95% CI = [nan, nan]
Fold 1, Repeat 8: AUC = 0.6190, 95% CI = [nan, nan]
Fold 1, Repeat 9: AUC = 0.5714, 95% CI = [nan, nan]
Fold 1, Repeat 10: AUC = 0.6190, 95% CI = [nan, nan]
Fold 2, Repeat 1: AUC = 0.1111, 95% CI = [nan, nan]
Fold 2, Repeat 2: AUC = 0.1111, 95% CI = [nan, nan]
Fold 2, Repeat 3: AUC = 0.1111, 95% CI = [nan, nan]
Fold 2, Repeat 4: AUC = 0.1111, 95% CI = [nan, nan]
Fold 2, Repeat 5: AUC = 0.8889, 95% CI = [nan, nan]
Fold 2, Repeat 6: AUC = 0.1111, 95% CI = [nan, nan]
Fold 2, Repeat 7: AUC = 0.1111, 95% CI = [nan, nan]
Fold 2, Repeat 8: AUC = 0.1111, 95% CI = [nan, nan]
Fold 2, Repeat 9: AUC = 0.1111, 95% CI = [nan, nan]
Fold 2, Rep

KeyboardInterrupt: 